In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

In [3]:
loader = PyPDFDirectoryLoader("../us_census")
documents = loader.load()

print("Documents loaded:",len(documents))

Documents loaded: 63


In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 316


In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("Embeddings loaded successfully")

c:\Users\adeku\Three-Drive\RAG\rag-project\ragenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\adeku\Three-Drive\RAG\rag-project\ragenv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Embeddings loaded successfully


In [6]:
vectorstore = FAISS.from_documents(documents, embeddings)



In [7]:
vectorstore.save_local("faiss_index")

In [8]:
vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

In [9]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [10]:
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.1-8b-instant",
    temperature=0
)

In [11]:
prompt = PromptTemplate.from_template("""
You are an intelligent PDF assistant.

Answer the question ONLY from the provided context.

If the answer is not present in the context, say:
"I could not find the answer in the provided documents."

Provide concise and accurate answers.

Context:
{context}

Question:
{input}

Answer:
""")

In [12]:
document_chain = create_stuff_documents_chain(llm, prompt)

retrieval_chain = create_retrieval_chain(
    retriever,
    document_chain
)

In [13]:
response = retrieval_chain.invoke({
    "input": "WHAT IS THE PRIVATE HEALTH INSURANCE COVERAGE BY STATE IN 2022?"
})

print("\nANSWER:\n")
print(response["answer"])

print("\nSOURCE DOCUMENTS:\n")

for i, doc in enumerate(response["context"]):
    print(f"Source {i+1}:")
    print(doc.page_content[:300])
    print("\n" + "="*50 + "\n")


ANSWER:

Utah and North Dakota reported the highest rate of private coverage (78.4 percent) in 2022, while New Mexico had the lowest private coverage rate (54.4 percent) (Figure 3).

SOURCE DOCUMENTS:

Source 1:
Health Insurance Coverage Status and Type 
by Geography: 2021 and 2022
American Community Survey Briefs
ACSBR-015
Issued September 2023
Douglas Conway and Breauna Branch
INTRODUCTION
Demographic shifts as well as economic and govern-
ment policy changes can affect people’s access to 
health coverage


Source 2:
10 U.S. Census Bureau
SUMMARY
The uninsured rate fell in 27 states 
(mainly states that had expanded 
Medicaid eligibility), while only 
Maine had an increase of 0.8 
percentage points. Only one state 
saw a decrease in public coverage 
(Rhode Island), while seven states 
experienced decreases in pr


Source 3:
2 U.S. Census Bureau
WHAT IS HEALTH INSURANCE COVERAGE?
This brief presents state-level estimates of health insurance coverage 
using data from the American Comm